# NeurIPS 2026 submission 18877: camera-ready analysis notebook

Six self-contained Colab cells, one per locked design, in run order. Each cell mounts Google Drive, verifies the SHA-256 of its inputs, runs once, and writes its outputs to the Drive folder named in its header; those outputs are the files archived under `artifacts/` in this package.

| Cell | Design (in `designs/`) | Output folder | Hardware |
|---|---|---|---|
| 1 | `design_18877_R1_M2_vs_strong_baselines.md` | `NeurIPS2026_3/rebuttal_R1/` | CPU, minutes |
| 2 | extends R1 Step 0 (precision readout) | `NeurIPS2026_3/rebuttal_R1/` | CPU, seconds |
| 3 | `design_18877_S2_sensitivity.md`, Part A | `NeurIPS2026_3/rebuttal_S2/` | CPU, minutes |
| 4 | `design_18877_S2_sensitivity.md`, Part B | `NeurIPS2026_3/rebuttal_S2/` | GPU (T4), ~70-100 min |
| 5 | `design_RV_S1_screening.md` | `NeurIPS2026_3/rv_screen_S1/` | CPU, minutes |
| 6 | `design_RV_F1_ladder.md` | `NeurIPS2026_3/rv_f1/` | GPU (T4), ~1 h |

Inputs expected on Drive: the V-Dem CSV from this package (`data/merged_completevars_balanced_zero_missing.csv`) under `NeurIPS2026_3/data/`; the submitted artifact pickles under `NeurIPS2026_3/results/regime_panar/vdem_v1h/` (cell 1 can also fetch the submitted bundle tarball directly); and, for cells 5 and 6, the realized-volatility panel `rv_dataset.csv` (eight equity indices, daily realized volatility), which is not redistributed with this package: readers should construct it from the source described in Son et al. (2023), cited in the paper. The SHA-256 of the file used is recorded in `artifacts/rv_screen_S1/results_RVS1_meta.json` and `artifacts/rv_f1/results_RVF1_meta.json` and is checked by both cells before any computation.

Cell outputs are preserved as the record of the runs that produced the archived artifacts. Exploratory and inventory cells from the working notebook have been removed.

## Cell 1: 18877-R1: panel-appropriate inference (Table 2 rows 1-3 and caption; artifacts/rebuttal_R1)

In [ ]:
# =============================================================================
# 18877-R1: M2 vs stratified linear / RW under country-clustered inference
# Single Colab cell. Design: design_18877_M2_vs_strong_baselines.md (LOCKED
# 2026-07-24; defaults: I5 excluded, NN_stratified excluded, claim levels §7).
# Stopping rule: one execution. Outputs -> Drive: NeurIPS2026_3/rebuttal_R1/
# =============================================================================
import os, io, json, tarfile, hashlib, urllib.request, datetime
import numpy as np, pandas as pd
from scipy import stats

from google.colab import drive
drive.mount('/content/drive', force_remount=False)

OUT_DIR = '/content/drive/MyDrive/NeurIPS2026_3/rebuttal_R1'
os.makedirs(OUT_DIR, exist_ok=True)

RNG_SEED = 20260724
B_BOOT   = 9999
ALPHA    = 0.05

TARBALL_URL = ('https://github.com/author0725/NeurIPS-18877/raw/main/'
               'panar-x-reproducibility.tar.gz')
PKL_MEMBER  = 'panar-x-reproducibility/artifacts/vdem_v1h_postanalysis.pkl'
FRONTIER_MEMBER = 'panar-x-reproducibility/artifacts/vdem_v1h_frontier.pkl'
PKL_SHA256  = '018a8d24d7af18f8f3f85cdc4bed8ed54166c0e55dc9170f91df29478d762e0a'
FRONTIER_SHA256 = '39fbb88dc739b905e62bed39f722386a3bb9c6531b2350739b7be273a13e5b03'

# ---- 1. Fetch canonical artifacts from the anonymous repo, verify SHA-256 ----
print('Downloading bundle ...')
raw = urllib.request.urlopen(TARBALL_URL, timeout=120).read()
tf = tarfile.open(fileobj=io.BytesIO(raw), mode='r:gz')

def _extract_verified(member, sha_expected):
    b = tf.extractfile(member).read()
    sha = hashlib.sha256(b).hexdigest()
    assert sha == sha_expected, f'SHA mismatch for {member}: {sha}'
    import pickle
    return pickle.loads(b), sha

post, post_sha = _extract_verified(PKL_MEMBER, PKL_SHA256)
frontier, frontier_sha = _extract_verified(FRONTIER_MEMBER, FRONTIER_SHA256)
print('SHA-256 verified for both artifacts.')

# ---- 2. Geometry checks (halt on mismatch) ----------------------------------
y   = np.asarray(post['targets'], dtype=float)          # realized dY
reg = np.asarray(post['regime_assignment'], dtype=int)
cid = np.asarray(post['country_id'])
P   = post['predictions']
assert len(y) == 1510, len(y)
assert [(reg == k).sum() for k in (0, 1, 2)] == [252, 621, 637]
assert len(np.unique(cid)) == 151
assert np.allclose(P['rw'], 0.0)
seeds = [np.asarray(s, dtype=float) for s in P['M2_per_seed']]
assert len(seeds) == 5
m2_ens = np.asarray(P['M2_mean'], dtype=float)
assert np.allclose(m2_ens, np.mean(seeds, axis=0))
print('Geometry checks passed.')

def se_of(pred):
    return (np.asarray(pred, dtype=float) - y) ** 2

SE = {  # squared errors, observation-aligned  [tag: ENS unless noted]
    'M2_ens'      : se_of(m2_ens),
    'strat_lin'   : se_of(P['linear_stratified_M1.5']),
    'pooled_lin'  : se_of(P['linear_pooled_M1.5']),
    'rw'          : se_of(P['rw']),
    'pooled_nn'   : se_of(P['NN_pooled_M1.5']),   # single-seed stored object
}
SE_seeds = [se_of(s) for s in seeds]              # [PR]

SCOPES = [('overall', np.ones_like(reg, bool)),
          ('regime0', reg == 0), ('regime1', reg == 1), ('regime2', reg == 2)]

# ---- 3. STEP 0: reconciliation of every printed Table 1/2 number ------------
# Printed values transcribed from submitted main.tex (verified 2026-07-24).
printed = []  # (label, scope, printed_value)
printed += [('pooled_lin_MSE',  s, v) for s, v in
            zip(['regime0','regime1','regime2','overall'], [0.00520,0.00487,0.00143,0.00347])]
printed += [('strat_lin_MSE',   s, v) for s, v in
            zip(['regime0','regime1','regime2','overall'], [0.00518,0.00465,0.00140,0.00337])]
printed += [('M2_MSE',          s, v) for s, v in
            zip(['regime0','regime1','regime2','overall'], [0.00515,0.00478,0.00136,0.00340])]
printed += [('pooled_nn_MSE',   s, v) for s, v in
            zip(['regime0','regime1','regime2','overall'], [0.00555,0.00481,0.00146,0.00352])]
printed += [('rw_MSE', 'overall', 0.00340)]
printed_dm = {  # naive paired DM, M2_ens vs pooled_nn, as printed in Table 2
    'regime0': (+2.20, 0.028), 'regime1': (+0.30, 0.765),
    'regime2': (+2.41, 0.016), 'overall': (+2.41, 0.016)}

key_of = {'pooled_lin_MSE':'pooled_lin', 'strat_lin_MSE':'strat_lin',
          'M2_MSE':'M2_ens', 'pooled_nn_MSE':'pooled_nn', 'rw_MSE':'rw'}
scope_mask = dict(SCOPES)

step0 = []
for label, scope, pv in printed:
    rec = float(SE[key_of[label]][scope_mask[scope]].mean())
    step0.append(dict(quantity=label, scope=scope, printed=pv,
                      recomputed=rec, recomputed_5dp=round(rec, 5),
                      match=(round(rec, 5) == pv), source='postanalysis.pkl'))

# Cross-check: 5-seed-mean baseline floats stored in the frontier pickle
for k, lbl in [('RW','rw_MSE'), ('pooled_linear_M1.5','pooled_lin_MSE'),
               ('stratified_linear_M1.5','strat_lin_MSE'),
               ('pooled_NN_M1.5','pooled_nn_MSE')]:
    step0.append(dict(quantity=lbl, scope='overall', printed=np.nan,
                      recomputed=float(frontier['baselines'][k]),
                      recomputed_5dp=round(float(frontier['baselines'][k]), 5),
                      match=np.nan, source='frontier.pkl baselines (cross-check)'))

def naive_dm(d):
    """Paired DM as in the submission: z = mean(d)/(sd(d)/sqrt(N)), normal p."""
    d = np.asarray(d, float)
    z = d.mean() / (d.std(ddof=1) / np.sqrt(len(d)))
    return z, 2 * stats.norm.sf(abs(z))

d_headline = SE['pooled_nn'] - SE['M2_ens']       # + favors M2 (paper convention)
for scope, mask in SCOPES:
    z, p = naive_dm(d_headline[mask])
    zp, pp = printed_dm[scope]
    step0.append(dict(quantity='DM_naive_M2_vs_pooledNN', scope=scope,
                      printed=f'{zp:+.2f} (p={pp})',
                      recomputed=f'{z:+.2f} (p={p:.3f})',
                      recomputed_5dp=np.nan,
                      match=(round(z, 2) == zp and round(p, 3) == pp),
                      source='postanalysis.pkl'))

step0_df = pd.DataFrame(step0)
step0_df.to_csv(f'{OUT_DIR}/results_18877_R1_step0_reconciliation.csv', index=False)
print('\n================ STEP 0: RECONCILIATION ================')
print(step0_df.to_string(index=False))

# ---- 4. Locked tests: country-block bootstrap-t + cluster-robust t ----------
rng = np.random.default_rng(RNG_SEED)

def cluster_stats(d, countries):
    """dbar, CR0 cluster SE for the observation mean, per-country sums/counts."""
    u, inv = np.unique(countries, return_inverse=True)
    S = np.bincount(inv, weights=d); n = np.bincount(inv).astype(float)
    N = n.sum(); dbar = S.sum() / N
    se = np.sqrt(((S - n * dbar) ** 2).sum()) / N
    return dbar, se, S, n, len(u)

def boot_t_test(d, countries, B, rng):
    dbar, se, S, n, G = cluster_stats(d, countries)
    t_obs = dbar / se
    idx = rng.integers(0, G, size=(B, G))
    S_m, n_m = S[idx], n[idx]
    Ntot = n_m.sum(1)
    dbar_b = S_m.sum(1) / Ntot
    se_b = np.sqrt(((S_m - n_m * dbar_b[:, None]) ** 2).sum(1)) / Ntot
    ok = se_b > 0
    t_b = (dbar_b[ok] - dbar) / se_b[ok]
    p_boot = (1 + (np.abs(t_b) >= abs(t_obs)).sum()) / (ok.sum() + 1)
    # secondary: t on unweighted country means, df = G-1
    m_g = S / n
    t_cl = m_g.mean() / (m_g.std(ddof=1) / np.sqrt(G))
    p_cl = 2 * stats.t.sf(abs(t_cl), df=G - 1)
    return dict(mean_d=dbar, t_boot=t_obs, p_boot=p_boot,
                t_cluster=t_cl, p_cluster=p_cl, n_countries=G)

FAMILIES = [  # (family, baseline_key, role)
    ('C1_M2_vs_stratlin',  'strat_lin',  'primary'),
    ('C2_M2_vs_rw',        'rw',         'primary'),
    ('C3_stratlin_vs_rw',  None,         'context'),   # special-cased below
    ('C4_M2_vs_poolednn',  'pooled_nn',  'context'),
]

rows = []
for family, base_key, role in FAMILIES:
    if family == 'C3_stratlin_vs_rw':
        d_full = SE['rw'] - SE['strat_lin']           # + favors stratified linear
        obj = 'det_vs_det'
    else:
        d_full = SE[base_key] - SE['M2_ens']          # + favors M2
        obj = 'ENS_vs_det' if base_key != 'pooled_nn' else 'ENS_vs_1seed(caveat)'
    for scope, mask in SCOPES:
        r = boot_t_test(d_full[mask], cid[mask], B_BOOT, rng)
        # per-seed [PR] consistency gate (not for C3)
        seed_ts, agree = [], np.nan
        if family != 'C3_stratlin_vs_rw':
            base_se = SE[base_key]
            for s_se in SE_seeds:
                ds = (base_se - s_se)[mask]
                db, se_, _, _, _ = cluster_stats(ds, cid[mask])
                seed_ts.append(db / se_ if se_ > 0 else np.nan)
            agree = int(sum(np.sign(t) == np.sign(r['t_boot'])
                            for t in seed_ts if np.isfinite(t)))
        rows.append(dict(family=family, role=role, scope=scope, object=obj,
                         N_obs=int(mask.sum()), **r,
                         seed_t_stats='|'.join(f'{t:+.2f}' for t in seed_ts) if seed_ts else '',
                         seed_sign_agree=f'{agree}/5' if seed_ts else ''))

res = pd.DataFrame(rows)

# Holm within C1 and C2 across the three regime tests (primary p = p_boot)
res['p_holm'] = np.nan
for family in ['C1_M2_vs_stratlin', 'C2_M2_vs_rw']:
    m = res.index[(res.family == family) & (res.scope != 'overall')]
    pvals = res.loc[m, 'p_boot'].to_numpy()
    order = np.argsort(pvals)
    adj = np.empty(3)
    running = 0.0
    for rank, oi in enumerate(order):
        running = max(running, (3 - rank) * pvals[oi])
        adj[oi] = min(1.0, running)
    res.loc[m, 'p_holm'] = adj

res.to_csv(f'{OUT_DIR}/results_18877_R1_dm_tests.csv', index=False)
print('\n================ LOCKED TESTS ================')
with pd.option_context('display.width', 200, 'display.max_columns', None):
    print(res.to_string(index=False))

# ---- 5. Pre-specified claim level (§7) --------------------------------------
def holm_sig(family, scope):
    v = res[(res.family == family) & (res.scope == scope)]['p_holm'].iloc[0]
    return v < ALPHA
c1_tails = [s for s in ('regime0', 'regime2') if holm_sig('C1_M2_vs_stratlin', s)]
c2_same  = [s for s in c1_tails if holm_sig('C2_M2_vs_rw', s)]
c2_tails = [s for s in ('regime0', 'regime2') if holm_sig('C2_M2_vs_rw', s)]
if c1_tails and c2_same:
    level, note = 'A', f'C1+C2 Holm-significant in: {c2_same}'
elif c2_tails and not c1_tails:
    level, note = 'B', f'C2 Holm-significant in {c2_tails}; C1 not'
else:
    level, note = 'C', 'no Holm-significant C1 tail result'
print(f'\nPre-specified claim level: {level} ({note})')

# ---- 6. Meta / provenance ---------------------------------------------------
meta = dict(design='design_18877_M2_vs_strong_baselines.md (locked 2026-07-24)',
            tarball_url=TARBALL_URL,
            postanalysis_sha256=post_sha, frontier_sha256=frontier_sha,
            rng_seed=RNG_SEED, B=B_BOOT, alpha=ALPHA,
            claim_level=level, claim_note=note,
            i5_included=False, nn_stratified_included=False,
            numpy=np.__version__, pandas=pd.__version__,
            timestamp_utc=datetime.datetime.now(datetime.timezone.utc).isoformat())
with open(f'{OUT_DIR}/results_18877_R1_meta.json', 'w') as fh:
    json.dump(meta, fh, indent=2)
print(f'\nWrote 3 files to {OUT_DIR}')

Mounted at /content/drive
SHA-256 verified for both artifacts.
Geometry checks passed.

================ STEP 0: RECONCILIATION ================
               quantity   scope         printed      recomputed  recomputed_5dp match                               source
         pooled_lin_MSE regime0          0.0052        0.005196         0.00520  True                     postanalysis.pkl
         pooled_lin_MSE regime1         0.00487        0.004867         0.00487  True                     postanalysis.pkl
         pooled_lin_MSE regime2         0.00143         0.00143         0.00143  True                     postanalysis.pkl
         pooled_lin_MSE overall         0.00347        0.003472         0.00347  True                     postanalysis.pkl
          strat_lin_MSE regime0         0.00518        0.005178         0.00518  True                     postanalysis.pkl
          strat_lin_MSE regime1         0.00465        0.004649         0.00465  True                     postanalysi

## Cell 3: Table 2 precision readout at three decimals (artifacts/rebuttal_R1/results_18877_R1_table2_precision.csv)

In [ ]:
# =============================================================================
# 18877 Table-2 precision readout: resolves the reviewer-flagged "duplicated"
# Overall row (+2.41, p=0.016 = Regime 2 row) at full precision.
# Reconciliation only — extends locked 18877-R1 Step 0; no new tests.
# Appends one small CSV next to the R1 outputs.
# =============================================================================
import hashlib, pickle
import numpy as np, pandas as pd
from scipy import stats
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

PKL = ('/content/drive/MyDrive/NeurIPS2026_3/results/regime_panar/vdem_v1h/'
       'vdem_v1h_postanalysis.pkl')
SHA = '018a8d24d7af18f8f3f85cdc4bed8ed54166c0e55dc9170f91df29478d762e0a'
OUT = '/content/drive/MyDrive/NeurIPS2026_3/rebuttal_R1/results_18877_R1_table2_precision.csv'

b = open(PKL, 'rb').read()
assert hashlib.sha256(b).hexdigest() == SHA
d = pickle.loads(b)
y, reg = np.asarray(d['targets'], float), np.asarray(d['regime_assignment'], int)
P = d['predictions']
diff = (np.asarray(P['NN_pooled_M1.5'], float) - y) ** 2 \
     - (np.asarray(P['M2_mean'], float) - y) ** 2          # + favors M2

rows = []
for scope, mask in [('overall', np.ones_like(reg, bool))] + \
                   [(f'regime{r}', reg == r) for r in (0, 1, 2)]:
    dd = diff[mask]
    z = dd.mean() / (dd.std(ddof=1) / np.sqrt(len(dd)))    # paper's naive DM
    p = 2 * stats.norm.sf(abs(z))
    rows.append(dict(scope=scope, N=len(dd),
                     dm_stat_full=f'{z:.10f}', p_full=f'{p:.10f}',
                     dm_2dp=round(z, 2), p_3dp=round(p, 3)))
t = pd.DataFrame(rows)
t.to_csv(OUT, index=False)
print(t.to_string(index=False))
ov, r2 = t.iloc[0], t.iloc[3]
print(f"\nOverall vs Regime 2 at full precision: "
      f"{ov.dm_stat_full} vs {r2.dm_stat_full} — "
      f"{'DISTINCT values coinciding at 2 dp (genuine coincidence, not a copy-paste)' if ov.dm_stat_full != r2.dm_stat_full else 'identical to 10 dp — investigate further before asserting coincidence'}")
print(f'Wrote {OUT}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
  scope    N dm_stat_full       p_full  dm_2dp  p_3dp
overall 1510 2.4111271968 0.0159033010    2.41  0.016
regime0  252 2.1961975783 0.0280778046    2.20  0.028
regime1  621 0.2993521006 0.7646714055    0.30  0.765
regime2  637 2.4068753560 0.0160896602    2.41  0.016

Overall vs Regime 2 at full precision: 2.4111271968 vs 2.4068753560 — DISTINCT values coinciding at 2 dp (genuine coincidence, not a copy-paste)
Wrote /content/drive/MyDrive/NeurIPS2026_3/rebuttal_R1/results_18877_R1_table2_precision.csv


## Cell 5: 18877-S2 Part A: diagnostic sensitivity grid, stability, stratification test (artifacts/rebuttal_S2)

In [ ]:
# =============================================================================
# 18877-S2 PART A (linear, CPU): diagnostic sensitivity grid (A1), stability
# across samples (A2), stratification-gain tests (A3).
# LOCKED design_18877_S2_sensitivity.md (2026-07-25). One execution.
# Outputs -> /content/drive/MyDrive/NeurIPS2026_3/rebuttal_S2/
# =============================================================================
import os, json, hashlib, datetime
import numpy as np, pandas as pd
from scipy import stats
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

DATA_PATH = ('/content/drive/MyDrive/NeurIPS2026_3/data/'
             'merged_completevars_balanced_zero_missing.csv')
DATA_SHA = '9c61549222c65491401c4552bc3311213dabaad396c2ee8aa32ccbbee722660f'
OUT_DIR = '/content/drive/MyDrive/NeurIPS2026_3/rebuttal_S2'
os.makedirs(OUT_DIR, exist_ok=True)

Y_VAR = 'v2xel_frefair'
X_VARS = ['v2csantimv', 'v2jureform', 'v2psorgs', 'v2mecorrpt']
L, TEST_YEARS = 5, 10
RNG_SEED_A3, B_BOOT, ALPHA = 20260815, 9999, 0.05
JK_SEED, N_JK = 20260817, 200
RIDGE_REL = 1e-8

# ---- 1. Data (ported from bundle regime_stratified_diagnostic.py; adds
#          cid/year per row for clustering and era splits) --------------------
assert hashlib.sha256(open(DATA_PATH, 'rb').read()).hexdigest() == DATA_SHA
df = pd.read_csv(DATA_PATH, keep_default_na=False, na_values=[''])
df = df.sort_values(['country_id', 'year']).reset_index(drop=True)

rows_tr, rows_te = [], []
for cid in df['country_id'].unique():
    sub = df[df['country_id'] == cid].sort_values('year').reset_index(drop=True)
    if len(sub) < L + TEST_YEARS + 1:
        continue
    thr = sub['year'].iloc[-TEST_YEARS]
    y_arr, x_arr, years = sub[Y_VAR].values, sub[X_VARS].values, sub['year'].values
    for i in range(L, len(sub)):
        yw, xw = y_arr[i-L:i+1], x_arr[i-L:i]
        if np.isnan(yw).any() or np.isnan(xw).any():
            continue
        r = dict(y_hist=yw[:-1], x_hist=xw, y_lag1=yw[-2], dY=yw[-1]-yw[-2],
                 cid=cid, year=years[i])
        (rows_tr if years[i] < thr else rows_te).append(r)

def arr(rows):
    return (np.stack([r['y_hist'] for r in rows]),
            np.stack([r['x_hist'] for r in rows]),
            np.array([r['y_lag1'] for r in rows]),
            np.array([r['dY'] for r in rows]),
            np.array([r['cid'] for r in rows]),
            np.array([r['year'] for r in rows]))
Yh_tr, Xh_tr, ylag1_tr, dY_tr, cid_tr, yr_tr = arr(rows_tr)
Yh_te, Xh_te, ylag1_te, dY_te, cid_te, yr_te = arr(rows_te)
assert len(dY_te) == 1510 and len(np.unique(cid_te)) == 151, \
    (len(dY_te), len(np.unique(cid_te)))
xm, xs = Xh_tr.mean(axis=0), Xh_tr.std(axis=0)
xs = np.where(xs < 1e-8, 1.0, xs)
Xh_tr, Xh_te = (Xh_tr - xm) / xs, (Xh_te - xm) / xs
print(f'train {len(dY_tr)}  test {len(dY_te)}  countries 151')

# slope naming: lag1 = t-1 (x_hist index -1) ... lag5 = t-5 (index 0)
SLOPES = [(v, l) for l in range(1, L + 1) for v in X_VARS]
def slope_cols(Xh):                     # (N, L, 4) -> (N, 20) ordered as SLOPES
    return np.concatenate([Xh[:, L - l, :] for l in range(1, L + 1)], axis=1)
Xs_tr, Xs_te = slope_cols(Xh_tr), slope_cols(Xh_te)
def design(Yh, Xs):
    return np.column_stack([np.ones(len(Yh)), Yh, Xs])   # 1 + 5 + 20
D_tr, D_te = design(Yh_tr, Xs_tr), design(Yh_te, Xs_te)
X_SLICE = slice(1 + L, 1 + L + len(SLOPES))

def ols_cr(D, y, cid):
    """OLS + CR0 country-clustered SEs. Returns beta, se."""
    G = D.T @ D
    G[np.diag_indices_from(G)] += RIDGE_REL * np.trace(G) / D.shape[1]
    Gi = np.linalg.inv(G)
    beta = Gi @ (D.T @ y)
    e = y - D @ beta
    meat = np.zeros_like(G)
    for c in np.unique(cid):
        m = cid == c
        s = D[m].T @ e[m]
        meat += np.outer(s, s)
    V = Gi @ meat @ Gi
    return beta, np.sqrt(np.clip(np.diag(V), 0, None))

# ---- 2. psi variants --------------------------------------------------------
def psi2(Yh): return Yh[:, -3:].mean(axis=1)             # mean(y_{t-1..t-3})
pc_feats_tr = np.column_stack([ylag1_tr, Xh_tr[:, -1, :]])
pcm, pcs = pc_feats_tr.mean(0), pc_feats_tr.std(0)
_, _, Vt = np.linalg.svd((pc_feats_tr - pcm) / pcs, full_matrices=False)
pc1 = Vt[0] * np.sign(Vt[0][0] if Vt[0][0] != 0 else 1.0)   # orient with y_lag1
def psi3(ylag1, Xh): return ((np.column_stack([ylag1, Xh[:, -1, :]]) - pcm) / pcs) @ pc1
PSIS = {'psi1_ylag1': (ylag1_tr, ylag1_te),
        'psi2_smooth': (psi2(Yh_tr), psi2(Yh_te)),
        'psi3_pc1': (psi3(ylag1_tr, Xh_tr), psi3(ylag1_te, Xh_te))}

def fit_variant(psi_tr, psi_te, pct_cuts, tr_D=D_tr, tr_y=dY_tr, tr_cid=cid_tr,
                tr_psi_override=None):
    """Fit pooled + per-regime models under given percentile cuts of training psi.
    Returns dict with betas/ses per regime, regimes arrays, test MSEs."""
    p_tr = psi_tr if tr_psi_override is None else tr_psi_override
    cuts = np.percentile(p_tr, pct_cuts)
    reg_tr = np.digitize(p_tr, cuts)
    reg_te = np.digitize(psi_te, cuts)
    R = len(pct_cuts) + 1
    b_pool, se_pool = ols_cr(tr_D, tr_y, tr_cid)
    betas, ses, preds = [], [], np.full(len(dY_te), np.nan)
    for r in range(R):
        m = reg_tr == r
        b, se = ols_cr(tr_D[m], tr_y[m], tr_cid[m])
        betas.append(b); ses.append(se)
        mt = reg_te == r
        if mt.any():
            preds[mt] = D_te[mt] @ b
    return dict(cuts=cuts, R=R, reg_tr=reg_tr, reg_te=reg_te,
                b_pool=b_pool, se_pool=se_pool,
                betas=np.array(betas), ses=np.array(ses),
                mse_strat=float(np.nanmean((dY_te - preds) ** 2)),
                mse_pool=float(((dY_te - D_te @ b_pool) ** 2).mean()),
                pred_strat=preds, pred_pool=D_te @ b_pool)

def inventory(v):
    """Sign reversals between outermost regimes among X slopes (2xCR-SE)."""
    b0, bL = v['betas'][0][X_SLICE], v['betas'][-1][X_SLICE]
    s0, sL = v['ses'][0][X_SLICE], v['ses'][-1][X_SLICE]
    rev = (np.sign(b0) != np.sign(bL)) & (np.abs(b0) > 2*s0) & (np.abs(bL) > 2*sL)
    return rev

PSORGS_L4 = SLOPES.index(('v2psorgs', 4))
def psorgs_flag(v):
    b0, bL = v['betas'][0][X_SLICE][PSORGS_L4], v['betas'][-1][X_SLICE][PSORGS_L4]
    s0, sL = v['ses'][0][X_SLICE][PSORGS_L4], v['ses'][-1][X_SLICE][PSORGS_L4]
    bp = v['b_pool'][X_SLICE][PSORGS_L4]
    present = bool(np.sign(b0) != np.sign(bL) and abs(b0) > 2*s0 and abs(bL) > 2*sL
                   and abs(bp) < min(abs(b0), abs(bL)) / 2)
    return present, bp, b0, bL

# ---- 3. A1: diagnostic grid -------------------------------------------------
VARIANTS = []
for pname in PSIS:
    for R in (2, 3, 4, 5):
        VARIANTS.append((f'{pname}_R{R}', pname, list(np.linspace(0, 100, R + 1)[1:-1])))
for pname in PSIS:                     # cut shifts, R=3 only, one cut at a time
    for tag, cuts in [('cutlo-5', [100/3 - 5, 200/3]), ('cutlo+5', [100/3 + 5, 200/3]),
                      ('cuthi-5', [100/3, 200/3 - 5]), ('cuthi+5', [100/3, 200/3 + 5])]:
        VARIANTS.append((f'{pname}_R3_{tag}', pname, cuts))

a1_rows, a1_summary = [], []
for vtag, pname, cuts in VARIANTS:
    v = fit_variant(*PSIS[pname], cuts)
    rev = inventory(v)
    pf, bp, b0, bL = psorgs_flag(v)
    canonical = vtag == 'psi1_ylag1_R3'
    if canonical:
        v_canon = v
    a1_summary.append(dict(variant=vtag, R=v['R'],
                           n_reversals=int(rev.sum()),
                           reversal_slopes=';'.join(f'{s}_lag{l}' for (s, l), rv
                                                    in zip(SLOPES, rev) if rv),
                           psorgs_l4_pattern=pf,
                           psorgs_l4_pooled=round(float(bp), 4),
                           psorgs_l4_R0=round(float(b0), 4),
                           psorgs_l4_Rlast=round(float(bL), 4),
                           mse_pooled=round(v['mse_pool'], 6),
                           mse_stratified=round(v['mse_strat'], 6)))
    for k, (s, l) in enumerate(SLOPES):
        row = dict(variant=vtag, covariate=s, lag=l,
                   pooled=round(float(v['b_pool'][X_SLICE][k]), 5),
                   reversal_outer=bool(rev[k]))
        for r in range(v['R']):
            row[f'beta_R{r}'] = round(float(v['betas'][r][X_SLICE][k]), 5)
            row[f'se_R{r}'] = round(float(v['ses'][r][X_SLICE][k]), 5)
        a1_rows.append(row)
pd.DataFrame(a1_rows).to_csv(f'{OUT_DIR}/results_S2_A1_diagnostic_grid.csv', index=False)
a1s = pd.DataFrame(a1_summary)
print('================ A1: DIAGNOSTIC GRID (summary) ================')
with pd.option_context('display.width', 220, 'display.max_columns', None):
    print(a1s.to_string(index=False))

# ---- 4. A2: stability across samples (canonical spec: psi1, R3, tertiles) ----
a2_rows = []
# (a) era splits
med_yr = int(np.median(yr_tr))
eras = [('train_first_half', yr_tr < med_yr), ('train_second_half', yr_tr >= med_yr),
        ('pre1996', yr_tr < 1996), ('pre2006', yr_tr < 2006), ('pre2011', yr_tr < 2011)]
for tag, m in eras:
    if m.sum() < 500:
        continue
    v = fit_variant(ylag1_tr[m], ylag1_te, [100/3, 200/3],
                    tr_D=D_tr[m], tr_y=dY_tr[m], tr_cid=cid_tr[m])
    pf, bp, b0, bL = psorgs_flag(v)
    a2_rows.append(dict(check='era', sample=tag, n=int(m.sum()),
                        psorgs_l4_pattern=pf, psorgs_l4_pooled=round(float(bp), 4),
                        psorgs_l4_R0=round(float(b0), 4), psorgs_l4_R2=round(float(bL), 4),
                        n_reversals=int(inventory(v).sum())))
# (b) country jackknife
jk = np.random.default_rng(JK_SEED)
countries = np.unique(cid_tr)
canon_signs0 = np.sign(v_canon['betas'][0][X_SLICE])
canon_signs2 = np.sign(v_canon['betas'][-1][X_SLICE])
keep0 = np.zeros(len(SLOPES)); keep2 = np.zeros(len(SLOPES)); psorgs_keep = 0
for _ in range(N_JK):
    cs = jk.choice(countries, size=len(countries)//2, replace=False)
    m = np.isin(cid_tr, cs)
    v = fit_variant(ylag1_tr[m], ylag1_te, [100/3, 200/3],
                    tr_D=D_tr[m], tr_y=dY_tr[m], tr_cid=cid_tr[m])
    keep0 += np.sign(v['betas'][0][X_SLICE]) == canon_signs0
    keep2 += np.sign(v['betas'][-1][X_SLICE]) == canon_signs2
    psorgs_keep += psorgs_flag(v)[0]
for k, (s, l) in enumerate(SLOPES):
    a2_rows.append(dict(check='jackknife', sample=f'{s}_lag{l}', n=N_JK,
                        frac_sign_R0=round(keep0[k] / N_JK, 3),
                        frac_sign_R2=round(keep2[k] / N_JK, 3)))
a2_rows.append(dict(check='jackknife', sample='psorgs_l4_full_pattern', n=N_JK,
                    frac_pattern=round(psorgs_keep / N_JK, 3)))
# (c) leave-one-covariate-out
canon_inv = inventory(v_canon)
for drop in X_VARS:
    keep_idx = [k for k, (s, l) in enumerate(SLOPES) if s != drop]
    Dm_tr = np.column_stack([np.ones(len(Yh_tr)), Yh_tr, Xs_tr[:, keep_idx]])
    xsl = slice(1 + L, 1 + L + len(keep_idx))
    b_p, _ = ols_cr(Dm_tr, dY_tr, cid_tr)
    reg = v_canon['reg_tr']
    bs, ss = [], []
    for r in range(3):
        m = reg == r
        b, se = ols_cr(Dm_tr[m], dY_tr[m], cid_tr[m])
        bs.append(b[xsl]); ss.append(se[xsl])
    b0, bL, s0, sL = bs[0], bs[-1], ss[0], ss[-1]
    rev = (np.sign(b0) != np.sign(bL)) & (np.abs(b0) > 2*s0) & (np.abs(bL) > 2*sL)
    canon_kept = [f'{SLOPES[k][0]}_lag{SLOPES[k][1]}'
                  for j, k in enumerate(keep_idx) if canon_inv[k] and rev[j]]
    canon_lost = [f'{SLOPES[k][0]}_lag{SLOPES[k][1]}'
                  for j, k in enumerate(keep_idx) if canon_inv[k] and not rev[j]]
    a2_rows.append(dict(check='loco', sample=f'drop_{drop}', n=int(len(dY_tr)),
                        reversals_kept=';'.join(canon_kept),
                        reversals_lost=';'.join(canon_lost),
                        n_reversals=int(rev.sum())))
a2 = pd.DataFrame(a2_rows)
a2.to_csv(f'{OUT_DIR}/results_S2_A2_stability.csv', index=False)
print('\n================ A2: STABILITY ================')
with pd.option_context('display.width', 220, 'display.max_columns', None):
    print(a2[a2.check != 'jackknife'].to_string(index=False))
    jkf = a2[(a2.check == 'jackknife')]
    print('\njackknife (sign-preservation fractions; canonical spec):')
    print(jkf.to_string(index=False))

# ---- 5. A3: stratification gain under clustered inference -------------------
def cluster_stats(d, countries):
    u, inv = np.unique(countries, return_inverse=True)
    S = np.bincount(inv, weights=d); n = np.bincount(inv).astype(float)
    N = n.sum(); dbar = S.sum() / N
    se = np.sqrt(((S - n * dbar) ** 2).sum()) / N
    return dbar, se, S, n, len(u)

def boot_t(d, countries, B, rng):
    dbar, se, S, n, G = cluster_stats(d, countries)
    t_obs = dbar / se
    idx = rng.integers(0, G, size=(B, G))
    S_m, n_m = S[idx], n[idx]; Ntot = n_m.sum(1)
    db = S_m.sum(1) / Ntot
    sb = np.sqrt(((S_m - n_m * db[:, None]) ** 2).sum(1)) / Ntot
    ok = sb > 0
    tb = (db[ok] - dbar) / sb[ok]
    return dbar, t_obs, (1 + (np.abs(tb) >= abs(t_obs)).sum()) / (ok.sum() + 1), G

rng = np.random.default_rng(RNG_SEED_A3)
d_full = (dY_te - v_canon['pred_pool']) ** 2 - (dY_te - v_canon['pred_strat']) ** 2
a3_rows = []
for scope, mask in [('overall', np.ones(len(dY_te), bool))] + \
                   [(f'regime{r}', v_canon['reg_te'] == r) for r in range(3)]:
    mo, t_o, p, G = boot_t(d_full[mask], cid_te[mask], B_BOOT, rng)
    a3_rows.append(dict(comparison='stratified_vs_pooled_linear', scope=scope,
                        N=int(mask.sum()), n_countries=G,
                        mse_pooled=float(((dY_te - v_canon['pred_pool'])**2)[mask].mean()),
                        mse_stratified=float(((dY_te - v_canon['pred_strat'])**2)[mask].mean()),
                        mean_d=mo, t_boot=t_o, p_boot=p))
a3 = pd.DataFrame(a3_rows)
a3.to_csv(f'{OUT_DIR}/results_S2_A3_stratification_tests.csv', index=False)
print('\n================ A3: STRATIFICATION GAIN (clustered) ================')
print(a3.to_string(index=False))

meta = dict(design='design_18877_S2_sensitivity.md (locked 2026-07-25) PART A',
            data_sha256=DATA_SHA, n_train=len(dY_tr), n_test=len(dY_te),
            n_variants=len(VARIANTS), jk_seed=JK_SEED, n_jk=N_JK,
            rng_seed_a3=RNG_SEED_A3, B=B_BOOT,
            canonical_psorgs_l4=dict(zip(['pattern', 'pooled', 'R0', 'R2'],
                                         [psorgs_flag(v_canon)[0]] +
                                         [round(float(x), 4) for x in psorgs_flag(v_canon)[1:]])),
            timestamp_utc=datetime.datetime.now(datetime.timezone.utc).isoformat())
with open(f'{OUT_DIR}/results_S2_metaA.json', 'w') as fh:
    json.dump(meta, fh, indent=2)
print(f'\nWrote Part A outputs to {OUT_DIR}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
train 6191  test 1510  countries 151
================ A1: DIAGNOSTIC GRID (summary) ================
               variant  R  n_reversals reversal_slopes  psorgs_l4_pattern  psorgs_l4_pooled  psorgs_l4_R0  psorgs_l4_Rlast  mse_pooled  mse_stratified
         psi1_ylag1_R2  2            0                              False              -0.0       -0.0214           0.0399    0.003472        0.003479
         psi1_ylag1_R3  3            0                              False              -0.0       -0.0281           0.0239    0.003472        0.003367
         psi1_ylag1_R4  4            0                              False              -0.0       -0.0271          -0.0297    0.003472        0.003388
         psi1_ylag1_R5  5            0                              False              -0.0       -0.0201          -0.0050    0.003472        0.003491
        psi2_sm

## Cell 7: 18877-S2 Part B: M2 retrained under four regime definitions, GPU (artifacts/rebuttal_S2)

In [ ]:
# =============================================================================
# 18877-S2 PART B (GPU): M2 sensitivity to regime definition — four variants,
# 5 seeds each, headline soft_only mode. LOCKED design_18877_S2_sensitivity.md.
# Purpose: robustness of the corrected null (no-upgrade rule, design §1).
# Checkpoint-resume permitted and logged. ~70-100 min on T4.
# Outputs -> /content/drive/MyDrive/NeurIPS2026_3/rebuttal_S2/
# =============================================================================
import os, json, hashlib, datetime, time
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
from scipy import stats
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

DATA_PATH = ('/content/drive/MyDrive/NeurIPS2026_3/data/'
             'merged_completevars_balanced_zero_missing.csv')
DATA_SHA = '9c61549222c65491401c4552bc3311213dabaad396c2ee8aa32ccbbee722660f'
OUT_DIR = '/content/drive/MyDrive/NeurIPS2026_3/rebuttal_S2'
CKPT = f'{OUT_DIR}/s2_checkpoints'
os.makedirs(CKPT, exist_ok=True)

Y_VAR = 'v2xel_frefair'
X_VARS = ['v2csantimv', 'v2jureform', 'v2psorgs', 'v2mecorrpt']
L, TEST_YEARS, N_X = 5, 10, 4
RNG_SEED_B, B_BOOT, ALPHA = 20260816, 9999, 0.05
EPOCHS, BATCH, LR, HID, WD = 300, 128, 3e-4, 32, 1e-4     # bundle headline cfg
TAU_INIT, LAM_PHI, LAM_Y, TGT_LY = 0.5, 0.005, 0.20, 0.55
SEEDS = [0, 1, 2, 3, 4]
DEV = 'cuda' if torch.cuda.is_available() else 'cpu'
RIDGE_REL = 1e-8
print(f'device: {DEV}')
resume_log = []

# ---- 1. Data (identical construction to Part A / bundle) --------------------
assert hashlib.sha256(open(DATA_PATH, 'rb').read()).hexdigest() == DATA_SHA
df = pd.read_csv(DATA_PATH, keep_default_na=False, na_values=[''])
df = df.sort_values(['country_id', 'year']).reset_index(drop=True)
rows_tr, rows_te = [], []
for cid in df['country_id'].unique():
    sub = df[df['country_id'] == cid].sort_values('year').reset_index(drop=True)
    if len(sub) < L + TEST_YEARS + 1:
        continue
    thr = sub['year'].iloc[-TEST_YEARS]
    y_arr, x_arr, years = sub[Y_VAR].values, sub[X_VARS].values, sub['year'].values
    for i in range(L, len(sub)):
        yw, xw = y_arr[i-L:i+1], x_arr[i-L:i]
        if np.isnan(yw).any() or np.isnan(xw).any():
            continue
        r = dict(y=yw[:-1], x=xw, ylag1=yw[-2], dY=yw[-1]-yw[-2], cid=cid)
        (rows_tr if years[i] < thr else rows_te).append(r)
def arr(rows):
    return (np.stack([r['y'] for r in rows])[:, :, None],
            np.stack([r['x'] for r in rows]),
            np.array([r['ylag1'] for r in rows]),
            np.array([r['dY'] for r in rows]),
            np.array([r['cid'] for r in rows]))
Yh_tr, Xh_tr, ylag1_tr, dY_tr, cid_tr = arr(rows_tr)
Yh_te, Xh_te, ylag1_te, dY_te, cid_te = arr(rows_te)
assert len(dY_te) == 1510 and len(np.unique(cid_te)) == 151
xm, xs = Xh_tr.mean(axis=0), Xh_tr.std(axis=0)
xs = np.where(xs < 1e-8, 1.0, xs)
Xh_tr, Xh_te = (Xh_tr - xm) / xs, (Xh_te - xm) / xs

def psi2(Yh): return Yh[:, -3:, 0].mean(axis=1)

# variant -> (psi_train, psi_test, percentile cut list)
VARIANTS = {
    'B_R2':   (ylag1_tr, ylag1_te, [50.0]),
    'B_R4':   (ylag1_tr, ylag1_te, [25.0, 50.0, 75.0]),
    'B_PSI2': (psi2(Yh_tr), psi2(Yh_te), [100/3, 200/3]),
    'B_CUT':  (ylag1_tr, ylag1_te, [100/3 + 5, 200/3 + 5]),   # both cuts +5pp
}

# ---- 2. M2 (bundle port; gate psi passed explicitly; R parameterized) -------
class PerEdgeNetM2(nn.Module):
    def __init__(self, Lm, hidden):
        super().__init__()
        self.lin1 = nn.Linear(Lm, hidden); self.lin2 = nn.Linear(hidden, 1)
    def forward(self, x):
        return self.lin2(F.relu(self.lin1(x))).squeeze(-1)

class M2_RegimeAware(nn.Module):
    def __init__(self, Lm, n_x, hidden, n_regimes, centers, tau_init=TAU_INIT):
        super().__init__()
        self.n_regimes, self.n_x = n_regimes, n_x
        self.f_yy = nn.ModuleList([PerEdgeNetM2(Lm, hidden) for _ in range(n_regimes)])
        self.f_xy = nn.ModuleList([nn.ModuleList([PerEdgeNetM2(Lm, hidden)
                                   for _ in range(n_x)]) for _ in range(n_regimes)])
        self.bias = nn.Parameter(torch.zeros(n_regimes))
        self.register_buffer('centers', torch.tensor(centers, dtype=torch.float32))
        self.log_tau = nn.Parameter(torch.tensor(np.log(tau_init), dtype=torch.float32))
    def gating(self, psi):
        tau = torch.exp(self.log_tau)
        d2 = (psi.unsqueeze(-1) - self.centers.unsqueeze(0)) ** 2
        return F.softmax(-d2 / tau, dim=-1)
    def per_regime_outputs(self, Yh, Xh):
        outs = []
        for r in range(self.n_regimes):
            o = self.bias[r] + self.f_yy[r](Yh[:, :, 0])
            for j in range(self.n_x):
                o = o + self.f_xy[r][j](Xh[:, :, j])
            outs.append(o)
        return torch.stack(outs, dim=-1)
    def forward(self, Yh, Xh, psi):
        return (self.gating(psi) * self.per_regime_outputs(Yh, Xh)).sum(dim=-1)

def per_regime_fn(model, r):
    def fn(Yh, Xh):
        o = model.bias[r] + model.f_yy[r](Yh[:, :, 0])
        for j in range(model.n_x):
            o = o + model.f_xy[r][j](Xh[:, :, j])
        return o
    return fn

def jac_softmax_Y(fn, Yh_b, Xh_b, tau_softmax=5.0):
    Yh_in = Yh_b.clone().requires_grad_(True)
    g = torch.autograd.grad(fn(Yh_in, Xh_b).sum(), Yh_in, create_graph=True)[0]
    n = g.flatten(1).norm(dim=1)
    return (torch.logsumexp(tau_softmax * n, dim=0) - np.log(n.shape[0])) / tau_softmax

def train_m2(model, Yh, Xh, psi, tgt):
    opt = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WD)
    Yt = torch.tensor(Yh, dtype=torch.float32, device=DEV)
    Xt = torch.tensor(Xh, dtype=torch.float32, device=DEV)
    Pt = torch.tensor(psi, dtype=torch.float32, device=DEV)
    Tt = torch.tensor(tgt, dtype=torch.float32, device=DEV)
    N = len(Yt)
    for ep in range(EPOCHS):
        perm = torch.randperm(N, device=DEV)
        for s in range(0, N, BATCH):
            idx = perm[s:s + BATCH]
            opt.zero_grad()
            loss = F.mse_loss(model(Yt[idx], Xt[idx], Pt[idx]), Tt[idx])
            spec = torch.zeros((), device=DEV)
            for r in range(model.n_regimes):
                spec = spec + F.relu(jac_softmax_Y(per_regime_fn(model, r),
                                                   Yt[idx], Xt[idx]) - TGT_LY) ** 2
            loss = loss + LAM_Y * spec + LAM_PHI * torch.exp(-model.log_tau) ** 2
            loss.backward(); opt.step()
    model.eval(); return model

# ---- 3. Train 4 variants x 5 seeds (checkpointed) ---------------------------
def ols(Xf, yf):
    G = Xf.T @ Xf; h = Xf.T @ yf
    G[np.diag_indices_from(G)] += RIDGE_REL * np.trace(G) / Xf.shape[1]
    return np.linalg.solve(G, h)

Dlin_tr = np.column_stack([np.ones(len(dY_tr)), Yh_tr[:, :, 0],
                           Xh_tr.reshape(len(dY_tr), -1)])
Dlin_te = np.column_stack([np.ones(len(dY_te)), Yh_te[:, :, 0],
                           Xh_te.reshape(len(dY_te), -1)])

results, variant_info = {}, {}
for vtag, (p_tr, p_te, pct) in VARIANTS.items():
    cuts = np.percentile(p_tr, pct)
    reg_tr = np.digitize(p_tr, cuts); reg_te = np.digitize(p_te, cuts)
    R = len(pct) + 1
    centers = [float(np.median(p_tr[reg_tr == r])) for r in range(R)]
    # variant-matched stratified linear
    strat = np.full(len(dY_te), np.nan)
    for r in range(R):
        b = ols(Dlin_tr[reg_tr == r], dY_tr[reg_tr == r])
        strat[reg_te == r] = Dlin_te[reg_te == r] @ b
    seed_preds = []
    for seed in SEEDS:
        ck = f'{CKPT}/{vtag}_s{seed}.pt'
        if os.path.exists(ck):
            d = torch.load(ck, map_location='cpu', weights_only=False)
            resume_log.append(f'{vtag}_s{seed}')
        else:
            torch.manual_seed(seed); np.random.seed(seed)
            m = M2_RegimeAware(L, N_X, HID, R, centers).to(DEV)
            t0 = time.time()
            train_m2(m, Yh_tr, Xh_tr, p_tr, dY_tr)
            with torch.no_grad():
                pr = m(torch.tensor(Yh_te, dtype=torch.float32, device=DEV),
                       torch.tensor(Xh_te, dtype=torch.float32, device=DEV),
                       torch.tensor(p_te, dtype=torch.float32, device=DEV)).cpu().numpy()
            d = dict(pred_te=pr, seed=seed, variant=vtag, R=R, centers=centers,
                     minutes=(time.time() - t0) / 60, state=m.cpu().state_dict())
            torch.save(d, ck)
        seed_preds.append(d['pred_te'])
    ens = np.mean(seed_preds, axis=0)
    results[vtag] = dict(ens=ens, seeds=seed_preds, strat=strat,
                         reg_te=reg_te, R=R)
    variant_info[vtag] = dict(R=R, cuts=[float(c) for c in cuts], centers=centers,
                              test_regime_counts=np.bincount(reg_te, minlength=R).tolist(),
                              mse_m2_ens=round(float(((dY_te - ens) ** 2).mean()), 6),
                              mse_strat=round(float(np.nanmean((dY_te - strat) ** 2)), 6),
                              mse_rw=round(float((dY_te ** 2).mean()), 6))
    print(vtag, variant_info[vtag])

# ---- 4. Clustered tests + Holm across variants ------------------------------
def cluster_stats(d, countries):
    u, inv = np.unique(countries, return_inverse=True)
    S = np.bincount(inv, weights=d); n = np.bincount(inv).astype(float)
    N = n.sum(); dbar = S.sum() / N
    return dbar, np.sqrt(((S - n * dbar) ** 2).sum()) / N, S, n, len(u)

def boot_t(d, countries, B, rng):
    dbar, se, S, n, G = cluster_stats(d, countries)
    t_obs = dbar / se
    idx = rng.integers(0, G, size=(B, G))
    S_m, n_m = S[idx], n[idx]; Ntot = n_m.sum(1)
    db = S_m.sum(1) / Ntot
    sb = np.sqrt(((S_m - n_m * db[:, None]) ** 2).sum(1)) / Ntot
    ok = sb > 0
    tb = (db[ok] - dbar) / sb[ok]
    return dbar, t_obs, (1 + (np.abs(tb) >= abs(t_obs)).sum()) / (ok.sum() + 1)

rng = np.random.default_rng(RNG_SEED_B)
rows = []
for fam, base_key in [('M2_vs_stratlin', 'strat'), ('M2_vs_rw', 'rw')]:
    for vtag in VARIANTS:
        rv = results[vtag]
        base_pred = rv['strat'] if base_key == 'strat' else np.zeros(len(dY_te))
        d_full = (dY_te - base_pred) ** 2 - (dY_te - rv['ens']) ** 2   # + favors M2
        mo, t_o, p = boot_t(d_full, cid_te, B_BOOT, rng)
        seed_ts = []
        for sp in rv['seeds']:
            ds = (dY_te - base_pred) ** 2 - (dY_te - sp) ** 2
            db, se, *_ = cluster_stats(ds, cid_te)
            seed_ts.append(db / se if se > 0 else np.nan)
        agree = int(sum(np.sign(t) == np.sign(t_o) for t in seed_ts if np.isfinite(t)))
        rows.append(dict(family=fam, variant=vtag, scope='overall', N=len(dY_te),
                         mean_d=mo, t_boot=t_o, p_boot=p,
                         seed_t='|'.join(f'{t:+.2f}' for t in seed_ts),
                         seed_agree=f'{agree}/5'))
res = pd.DataFrame(rows)
res['p_holm'] = np.nan
for fam in res.family.unique():
    m = res.index[res.family == fam]
    pv = res.loc[m, 'p_boot'].to_numpy(); order = np.argsort(pv); k = len(pv)
    adj = np.empty(k); running = 0.0
    for rank, oi in enumerate(order):
        running = max(running, (k - rank) * pv[oi]); adj[oi] = min(1.0, running)
    res.loc[m, 'p_holm'] = adj
res.to_csv(f'{OUT_DIR}/results_S2_B_m2_variants.csv', index=False)
print('\n================ PART B: M2 VARIANT TESTS (overall) ================')
with pd.option_context('display.width', 220, 'display.max_columns', None):
    print(res.to_string(index=False))
sig_adv = res[(res.p_holm < ALPHA) & (res.mean_d > 0)]
sig_dis = res[(res.p_holm < ALPHA) & (res.mean_d < 0)]
if len(sig_adv):
    print('\nNO-UPGRADE RULE (design §1): rows with p_holm < 0.05 and mean_d > 0 '
          'are EXPLORATORY and cannot enter the rebuttal as confirmatory claims.')
elif len(sig_dis):
    print('\nM2 significantly UNDERPERFORMS in some variants after Holm '
          f'({", ".join(sig_dis.variant + "/" + sig_dis.family)}) — report as such.')
else:
    print('\nAll variants null after Holm — the corrected equivalence is not an '
          'artifact of the submitted regime definition.')

meta = dict(design='design_18877_S2_sensitivity.md (locked 2026-07-25) PART B',
            data_sha256=DATA_SHA, variants=variant_info,
            epochs=EPOCHS, batch=BATCH, lr=LR, hidden=HID, seeds=SEEDS,
            tau_init=TAU_INIT, lam_phi=LAM_PHI, lam_Y=LAM_Y, target_LY=TGT_LY,
            rng_seed_tests=RNG_SEED_B, B=B_BOOT,
            resumed_checkpoints=resume_log, device=DEV, torch=torch.__version__,
            timestamp_utc=datetime.datetime.now(datetime.timezone.utc).isoformat())
with open(f'{OUT_DIR}/results_S2_metaB.json', 'w') as fh:
    json.dump(meta, fh, indent=2)
print(f'\nWrote Part B outputs to {OUT_DIR}')

Mounted at /content/drive
device: cuda
B_R2 {'R': 2, 'cuts': [0.41], 'centers': [0.1, 0.81599998], 'test_regime_counts': [563, 947], 'mse_m2_ens': 0.003393, 'mse_strat': 0.003479, 'mse_rw': 0.003396}
B_R4 {'R': 4, 'cuts': [0.1005, 0.41, 0.81599998], 'centers': [0.0, 0.22400001, 0.58999997, 0.92900002], 'test_regime_counts': [170, 393, 497, 450], 'mse_m2_ens': 0.003393, 'mse_strat': 0.003388, 'mse_rw': 0.003396}
B_PSI2 {'R': 3, 'cuts': [0.16477777777777783, 0.6486666733333333], 'centers': [0.032000002, 0.38800000333333334, 0.9084999950000001], 'test_regime_counts': [228, 601, 681], 'mse_m2_ens': 0.003389, 'mse_strat': 0.00347, 'mse_rw': 0.003396}
B_CUT {'R': 3, 'cuts': [0.242, 0.75400001], 'centers': [0.041999999, 0.47099999, 0.92050001], 'test_regime_counts': [339, 637, 534], 'mse_m2_ens': 0.003377, 'mse_strat': 0.003427, 'mse_rw': 0.003396}

================ PART B: M2 VARIANT TESTS (overall) ================
        family variant   scope    N    mean_d    t_boot  p_boot             

## Cell 9: RV-S1: realized-volatility screen, two gates (artifacts/rv_screen_S1)

In [ ]:
# =============================================================================
# RV-S1: Second-panel screening — realized volatility (LOCKED design
# design_RV_S1_screening.md, confirmed 2026-07-25). Gates only; NO M2 anywhere.
# One execution. Outputs -> /content/drive/MyDrive/NeurIPS2026_3/rv_screen_S1/
# =============================================================================
import os, json, hashlib, datetime
import numpy as np, pandas as pd
from scipy import stats

from google.colab import drive
drive.mount('/content/drive', force_remount=False)

DATA_PATH = '/content/drive/MyDrive/GNAVAR/data/rv_dataset.csv'
DATA_SHA  = '9f6065e3ac046a1a18512668fa71457dba5dc51bc1cb6e802a1441cd96a0098c'
OUT_DIR   = '/content/drive/MyDrive/NeurIPS2026_3/rv_screen_S1'
os.makedirs(OUT_DIR, exist_ok=True)

RNG_SEED, B_BOOT, BLOCK, P_LAGS, ALPHA = 20260801, 9999, 25, 4, 0.05
RIDGE_REL = 1e-8   # implementation detail: tiny relative ridge on Gram
                   # diagonals guarantees invertibility if an asset drops out
                   # of a regime in a bootstrap resample; bias negligible.

# ---- 1. Load, verify, transform ---------------------------------------------
raw_bytes = open(DATA_PATH, 'rb').read()
assert hashlib.sha256(raw_bytes).hexdigest() == DATA_SHA, 'SHA mismatch — halt'
df = pd.read_csv(DATA_PATH, keep_default_na=False, na_values=[''])
ASSETS = ['.SPX', '.GDAXI', '.FCHI', '.FTSE', '.OMXSPI', '.N225', '.KS11', '.HSI']
assert df.shape == (2615, 9) and list(df.columns[1:]) == ASSETS
V = df[ASSETS].to_numpy(float)                     # (T, A) daily vol
assert (V > 0).all() and not np.isnan(V).any()
T, A = V.shape
Y = np.log(V)                                      # y_it
# spillover covariate: leave-one-out cross-asset mean of y at each day
Xco = (Y.sum(axis=1, keepdims=True) - Y) / (A - 1)

# usable targets: t = P_LAGS .. T-1 ; Δy_t = y_t − y_{t-1}
t_idx = np.arange(P_LAGS, T)
DY = Y[t_idx] - Y[t_idx - 1]                       # (Tu, A)
LAG_Y = np.stack([Y[t_idx - l] for l in range(1, P_LAGS + 1)], axis=2)   # (Tu,A,4)
LAG_X = np.stack([Xco[t_idx - l] for l in range(1, P_LAGS + 1)], axis=2) # (Tu,A,4)
Tu = len(t_idx)

TEST_DAYS = 523                                    # final 20% of the 2,615 days
test_mask_day = t_idx >= (T - TEST_DAYS)
train_mask_day = ~test_mask_day
n_test_days, n_train_days = int(test_mask_day.sum()), int(train_mask_day.sum())
print(f'Usable days: {Tu} (train {n_train_days}, test {n_test_days}) × {A} assets')

# regimes: tertiles of y_{t-1} pooled across assets, TRAINING portion only
ylag1 = LAG_Y[:, :, 0]                             # (Tu, A)
cuts = np.quantile(ylag1[train_mask_day].ravel(), [1/3, 2/3])
REG = np.digitize(ylag1, cuts)                     # (Tu, A) in {0,1,2}
print('regime counts (train):', np.bincount(REG[train_mask_day].ravel(), minlength=3))
print('regime counts (test): ', np.bincount(REG[test_mask_day].ravel(), minlength=3))
print('per-regime × asset train counts:')
print(pd.DataFrame({a: [int(((REG[train_mask_day] == r)[:, j]).sum()) for r in range(3)]
                    for j, a in enumerate(ASSETS)}, index=['R0', 'R1', 'R2']).to_string())

# design matrix per observation: [4 own lags | 4 spillover lags | 8 asset dummies]
K_SLOPES, K = 2 * P_LAGS, 2 * P_LAGS + A
def design_rows(day_sel, regime=None):
    """Rows (obs) for selected usable-day mask; optional regime filter."""
    dsel = np.where(day_sel)[0]
    Xb = np.concatenate([LAG_Y[dsel].reshape(-1, P_LAGS),
                         LAG_X[dsel].reshape(-1, P_LAGS)], axis=1)  # (n, 8)
    dumm = np.tile(np.eye(A), (len(dsel), 1))
    Xf = np.concatenate([Xb, dumm], axis=1)
    yf = DY[dsel].ravel()
    day_of = np.repeat(dsel, A)
    reg_of = REG[dsel].ravel()
    if regime is not None:
        m = reg_of == regime
        return Xf[m], yf[m], day_of[m]
    return Xf, yf, day_of

def ols(Xf, yf):
    G = Xf.T @ Xf; h = Xf.T @ yf
    G[np.diag_indices_from(G)] += RIDGE_REL * np.trace(G) / K
    return np.linalg.solve(G, h)

# ---- 2. Fit models on training ----------------------------------------------
Xtr, ytr, _ = design_rows(train_mask_day)
beta_pooled = ols(Xtr, ytr)
beta_reg = []
for r in range(3):
    Xr, yr, _ = design_rows(train_mask_day, regime=r)
    beta_reg.append(ols(Xr, yr))
beta_reg = np.array(beta_reg)                      # (3, K)

# test predictions
Xte, yte, day_te = design_rows(test_mask_day)
reg_te = REG[test_mask_day].ravel()
pred_pooled = Xte @ beta_pooled
pred_strat  = np.einsum('nk,nk->n', Xte, beta_reg[reg_te])
se = {'rw': yte ** 2,
      'pooled_lin': (yte - pred_pooled) ** 2,
      'strat_lin':  (yte - pred_strat) ** 2}

# ---- 3. Gate 1: time-block bootstrap-t on test loss differentials -----------
rng = np.random.default_rng(RNG_SEED)
test_days_sorted = np.sort(np.unique(day_te))
day_pos = {d: i for i, d in enumerate(test_days_sorted)}
pos_te = np.array([day_pos[d] for d in day_te])

def day_mean_series(d_obs):
    s = np.zeros(n_test_days); c = np.zeros(n_test_days)
    np.add.at(s, pos_te, d_obs); np.add.at(c, pos_te, 1)
    return s / c

def bbt_stat(series):
    nfull = (len(series) // BLOCK) * BLOCK
    bm = series[:nfull].reshape(-1, BLOCK).mean(axis=1)
    return series.mean(), bm.std(ddof=1) / np.sqrt(len(bm))

def block_boot_p(series, B, rng):
    mean_o, se_o = bbt_stat(series); t_obs = mean_o / se_o
    nblk = int(np.ceil(len(series) / BLOCK))
    starts = rng.integers(0, len(series) - BLOCK + 1, size=(B, nblk))
    idx = (starts[:, :, None] + np.arange(BLOCK)[None, None, :]).reshape(B, -1)[:, :len(series)]
    res = series[idx]                              # (B, n_days)
    mean_b = res.mean(axis=1)
    nfull = (res.shape[1] // BLOCK) * BLOCK
    bm = res[:, :nfull].reshape(B, -1, BLOCK).mean(axis=2)
    se_b = bm.std(axis=1, ddof=1) / np.sqrt(bm.shape[1])
    ok = se_b > 0
    t_b = (mean_b[ok] - mean_o) / se_b[ok]
    p = (1 + (np.abs(t_b) >= abs(t_obs)).sum()) / (ok.sum() + 1)
    return mean_o, t_obs, p

def hac_dm_per_asset(d_obs):
    out = []
    for j, a in enumerate(ASSETS):
        d = d_obs[np.arange(len(d_obs)) % A == j]  # rows are day-major, asset-minor
        g0 = np.var(d, ddof=0); lrv = g0
        for l in range(1, BLOCK + 1):
            g = np.mean((d[l:] - d.mean()) * (d[:-l] - d.mean()))
            lrv += 2 * (1 - l / (BLOCK + 1)) * g
        stat = d.mean() / np.sqrt(lrv / len(d))
        out.append(f'{a}:{stat:+.2f}')
    return '|'.join(out)

g1_rows = []
for name, base, mdl in [('pooled_lin_vs_rw', 'rw', 'pooled_lin'),
                        ('strat_lin_vs_rw',  'rw', 'strat_lin'),
                        ('strat_vs_pooled',  'pooled_lin', 'strat_lin')]:
    d_obs = se[base] - se[mdl]                     # + favors the model
    mo, t_o, p = block_boot_p(day_mean_series(d_obs), B_BOOT, rng)
    g1_rows.append(dict(comparison=name, role=('GATE' if name == 'pooled_lin_vs_rw' else 'context'),
                        n_obs=len(d_obs), n_days=n_test_days,
                        mse_base=se[base].mean(), mse_model=se[mdl].mean(),
                        mean_d=mo, t_boot=t_o, p_boot=p,
                        per_asset_hac_dm=hac_dm_per_asset(d_obs)))
g1 = pd.DataFrame(g1_rows)
gate1_pass = bool((g1.iloc[0].p_boot < ALPHA) and (g1.iloc[0].mean_d > 0))

# ---- 4. Gate 2: regime heterogeneity via block-bootstrap Wald ---------------
# Per-day Gram/moment stacks per regime over TRAINING days (multiplicity trick:
# a block resample of days = integer-weighted normal equations).
train_days_sorted = np.where(train_mask_day)[0]
nT = len(train_days_sorted)
G_stack = np.zeros((3, nT, K, K)); h_stack = np.zeros((3, nT, K))
for i, d in enumerate(train_days_sorted):
    onehot = np.zeros(Tu, bool); onehot[d] = True
    for r in range(3):
        Xr, yr, _ = design_rows(onehot, regime=r)
        if len(yr):
            G_stack[r, i] = Xr.T @ Xr; h_stack[r, i] = Xr.T @ yr

def fit_from_weights(w):                            # w: (nT,) multiplicities
    betas = np.zeros((3, K))
    for r in range(3):
        G = np.tensordot(w, G_stack[r], axes=1)
        h = w @ h_stack[r]
        G[np.diag_indices_from(G)] += RIDGE_REL * np.trace(G) / K
        betas[r] = np.linalg.solve(G, h)
    return betas

d_obs16 = np.concatenate([beta_reg[0, :K_SLOPES] - beta_reg[1, :K_SLOPES],
                          beta_reg[1, :K_SLOPES] - beta_reg[2, :K_SLOPES]])
nblk_tr = int(np.ceil(nT / BLOCK))
starts = rng.integers(0, nT - BLOCK + 1, size=(B_BOOT, nblk_tr))
idx_tr = (starts[:, :, None] + np.arange(BLOCK)[None, None, :]).reshape(B_BOOT, -1)[:, :nT]
d_boot = np.zeros((B_BOOT, 2 * K_SLOPES)); beta_boot = np.zeros((B_BOOT, 3, K_SLOPES))
for b in range(B_BOOT):
    w = np.bincount(idx_tr[b], minlength=nT).astype(float)
    bb = fit_from_weights(w)
    beta_boot[b] = bb[:, :K_SLOPES]
    d_boot[b] = np.concatenate([bb[0, :K_SLOPES] - bb[1, :K_SLOPES],
                                bb[1, :K_SLOPES] - bb[2, :K_SLOPES]])
Sigma = np.cov(d_boot, rowvar=False)
W = float(d_obs16 @ np.linalg.solve(Sigma, d_obs16))
p_wald = float(stats.chi2.sf(W, df=2 * K_SLOPES))
cond = float(np.linalg.cond(Sigma))

slope_names = [f'own_lag{l}' for l in range(1, 5)] + [f'spill_lag{l}' for l in range(1, 5)]
se_boot = beta_boot.std(axis=0, ddof=1)            # (3, 8)
g2_rows, reversals = [], []
for k, nm in enumerate(slope_names):
    b0, b1, b2 = beta_reg[:, k]
    s0, s1, s2 = se_boot[:, k]
    rev = bool(np.sign(b0) != np.sign(b2) and abs(b0) > 2 * s0 and abs(b2) > 2 * s2
               and b0 != 0 and b2 != 0)
    if rev: reversals.append(nm)
    g2_rows.append(dict(slope=nm, pooled=beta_pooled[k],
                        beta_R0=b0, beta_R1=b1, beta_R2=b2,
                        se_R0=s0, se_R1=s1, se_R2=s2, sign_reversal_R0_R2=rev))
g2 = pd.DataFrame(g2_rows)
gate2_full    = bool(p_wald < ALPHA and len(reversals) > 0)
gate2_partial = bool(p_wald < ALPHA and len(reversals) == 0)

# ---- 5. Report and write ----------------------------------------------------
print('\n================ GATE 1: HEADROOM ================')
with pd.option_context('display.width', 220, 'display.max_columns', None):
    print(g1.to_string(index=False))
print(f'\nGATE 1: {"PASS" if gate1_pass else "FAIL"}')
print('\n================ GATE 2: REGIME STRUCTURE ================')
print(g2.to_string(index=False))
print(f'\nWald W = {W:.2f} on chi2({2*K_SLOPES}), p = {p_wald:.2e} '
      f'(Sigma condition number {cond:.1e})')
print(f'Sign reversals (R0 vs R2): {reversals if reversals else "none"}')
verdict = ('PASS' if gate2_full else 'PARTIAL PASS (heterogeneity without sign reversal '
           '— per design §5, whether this justifies the full experiment is Valya\'s call)'
           if gate2_partial else 'FAIL')
print(f'GATE 2: {verdict}')

g1.to_csv(f'{OUT_DIR}/results_RVS1_gate1_headroom.csv', index=False)
g2.to_csv(f'{OUT_DIR}/results_RVS1_gate2_diagnostic.csv', index=False)
meta = dict(design='design_RV_S1_screening.md (locked 2026-07-25)',
            data_path=DATA_PATH, data_sha256=DATA_SHA,
            assets=ASSETS, T_days=T, usable_days=Tu,
            train_days=n_train_days, test_days=n_test_days,
            regime_cuts=[float(c) for c in cuts],
            rng_seed=RNG_SEED, B=B_BOOT, block_len=BLOCK, p_lags=P_LAGS,
            ridge_rel=RIDGE_REL, wald_W=W, wald_p=p_wald, sigma_cond=cond,
            gate1_pass=gate1_pass, gate2_verdict=verdict, reversals=reversals,
            numpy=np.__version__, pandas=pd.__version__,
            timestamp_utc=datetime.datetime.now(datetime.timezone.utc).isoformat())
with open(f'{OUT_DIR}/results_RVS1_meta.json', 'w') as fh:
    json.dump(meta, fh, indent=2)
print(f'\nWrote 3 files to {OUT_DIR}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Usable days: 2611 (train 2088, test 523) × 8 assets
regime counts (train): [5568 5568 5568]
regime counts (test):  [2919  989  276]
per-regime × asset train counts:
    .SPX  .GDAXI  .FCHI  .FTSE  .OMXSPI  .N225  .KS11  .HSI
R0   886     402    374    590      824    708   1035   749
R1   575     746    761    712      632    804    581   757
R2   627     940    953    786      632    576    472   582

================ GATE 1: HEADROOM ================
      comparison    role  n_obs  n_days  mse_base  mse_model    mean_d    t_boot  p_boot                                                                                 per_asset_hac_dm
pooled_lin_vs_rw    GATE   4184     523  0.103898   0.074620  0.029278  9.356729  0.0001 .SPX:+1.61|.GDAXI:+7.15|.FCHI:+5.62|.FTSE:+7.92|.OMXSPI:+3.16|.N225:+6.01|.KS11:+6.66|.HSI:+5.24
 strat_lin_vs_rw context   4184     523  0

## Cell 11: RV-F1: realized-volatility confirmatory run and certificate audit, GPU (artifacts/rv_f1)

In [ ]:
# =============================================================================
# RV-F1: M2 on the realized-volatility panel (LOCKED design_RV_F1_ladder.md,
# 2026-07-25). Decisive baseline: interaction-augmented pooled linear (B2).
# One execution (checkpoint-resume permitted and logged; never retrains a
# completed run). GPU (T4) recommended. Outputs -> NeurIPS2026_3/rv_f1/
# =============================================================================
import os, json, hashlib, datetime, time, glob
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
from scipy import stats

from google.colab import drive
drive.mount('/content/drive', force_remount=False)

DATA_PATH = '/content/drive/MyDrive/GNAVAR/data/rv_dataset.csv'
DATA_SHA  = '9f6065e3ac046a1a18512668fa71457dba5dc51bc1cb6e802a1441cd96a0098c'
FRONTIER_PATH = ('/content/drive/MyDrive/NeurIPS2026_3/results/regime_panar/'
                 'vdem_v1h/vdem_v1h_frontier.pkl')
FRONTIER_SHA  = '39fbb88dc739b905e62bed39f722386a3bb9c6531b2350739b7be273a13e5b03'
OUT_DIR = '/content/drive/MyDrive/NeurIPS2026_3/rv_f1'
CKPT    = f'{OUT_DIR}/rvf1_checkpoints'
os.makedirs(CKPT, exist_ok=True)

RNG_SEED_TESTS, B_BOOT, BLOCK, P_LAGS, ALPHA = 20260810, 9999, 25, 4, 0.05
RIDGE_REL = 1e-8
SCREEN_B1_MSE = 0.074620            # RV-S1 cross-check target (6 dp)
EPOCHS, BATCH, WD = 300, 128, 1e-4  # bundle defaults (soft_only headline)
TAU_INIT, LAM_PHI, LAM_Y, TGT_LY = 0.5, 0.005, 0.20, 0.55
GRID = [(lr, hid) for lr in (1e-3, 3e-4) for hid in (16, 32)]  # pre-registered
SEEDS = [0, 1, 2, 3, 4]
DEV = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'device: {DEV}')
resume_log = []

# ---- 1. Data (identical to RV-S1) -------------------------------------------
assert hashlib.sha256(open(DATA_PATH, 'rb').read()).hexdigest() == DATA_SHA
df = pd.read_csv(DATA_PATH, keep_default_na=False, na_values=[''])
ASSETS = ['.SPX', '.GDAXI', '.FCHI', '.FTSE', '.OMXSPI', '.N225', '.KS11', '.HSI']
assert df.shape == (2615, 9) and list(df.columns[1:]) == ASSETS
V = df[ASSETS].to_numpy(float); T, A = V.shape
assert (V > 0).all() and not np.isnan(V).any()
Y = np.log(V)
Xco = (Y.sum(axis=1, keepdims=True) - Y) / (A - 1)
t_idx = np.arange(P_LAGS, T)
DY = Y[t_idx] - Y[t_idx - 1]
LAG_Y = np.stack([Y[t_idx - l] for l in range(P_LAGS, 0, -1)], axis=2)   # oldest->newest
LAG_X = np.stack([Xco[t_idx - l] for l in range(P_LAGS, 0, -1)], axis=2)
Tu = len(t_idx)
TEST_DAYS = 523
test_mask_day = t_idx >= (T - TEST_DAYS); train_mask_day = ~test_mask_day
ylag1 = LAG_Y[:, :, -1]                                                   # y_{t-1}
cuts = np.quantile(ylag1[train_mask_day].ravel(), [1/3, 2/3])
REG = np.digitize(ylag1, cuts)
centers = np.quantile(ylag1[train_mask_day].ravel(), [1/6, 1/2, 5/6])
print(f'regime cuts {cuts.round(4)}  gate centers {centers.round(4)}')

# Row layout: LAG_Y[dsel] is (n_days, A, L) with lags oldest->newest, so a
# plain reshape gives day-major, asset-minor rows of shape (n, L); [:, :, None]
# appends the channel dim so Y_hist[:, :, 0] and Y_hist[:, -1, 0] (= y_{t-1})
# match the bundle's layout exactly (newest lag last; asserted below).
def build(day_sel):
    dsel = np.where(day_sel)[0]
    Yh = LAG_Y[dsel].reshape(-1, P_LAGS)          # (n_days*A, L), day-major
    Xh = LAG_X[dsel].reshape(-1, P_LAGS)
    y  = DY[dsel].ravel()
    day_of = np.repeat(dsel, A)
    asset_of = np.tile(np.arange(A), len(dsel))
    reg_of = REG[dsel].ravel()
    return Yh[:, :, None], Xh[:, :, None], y, day_of, asset_of, reg_of

Yh_tr, Xh_tr, y_tr, day_tr, ast_tr, reg_tr = build(train_mask_day)
Yh_te, Xh_te, y_te, day_te, ast_te, reg_te = build(test_mask_day)
assert np.allclose(Yh_tr[:, -1, 0], ylag1[train_mask_day].ravel())  # newest-last check
n_test_days = TEST_DAYS
print(f'train obs {len(y_tr)}  test obs {len(y_te)}  '
      f'test regime counts {np.bincount(reg_te, minlength=3)}')

# ---- 2. Deterministic baselines --------------------------------------------
def ols(Xf, yf):
    G = Xf.T @ Xf; h = Xf.T @ yf
    G[np.diag_indices_from(G)] += RIDGE_REL * np.trace(G) / Xf.shape[1]
    return np.linalg.solve(G, h)

def dm_base(Yh, Xh, ast):                          # 8 slopes + asset dummies
    base = np.concatenate([Yh[:, :, 0], Xh[:, :, 0]], axis=1)
    return np.concatenate([base, np.eye(A)[ast]], axis=1)

def dm_inter(Yh, Xh, ast):                         # + psi x each base regressor
    base = np.concatenate([Yh[:, :, 0], Xh[:, :, 0]], axis=1)
    psi = Yh[:, -1, 0:1]
    return np.concatenate([base, psi * base, np.eye(A)[ast]], axis=1)

Xb_tr, Xb_te = dm_base(Yh_tr, Xh_tr, ast_tr), dm_base(Yh_te, Xh_te, ast_te)
Xi_tr, Xi_te = dm_inter(Yh_tr, Xh_tr, ast_tr), dm_inter(Yh_te, Xh_te, ast_te)
pred = {'rw': np.zeros_like(y_te),
        'B1_pooled': Xb_te @ ols(Xb_tr, y_tr),
        'B2_interaction': Xi_te @ ols(Xi_tr, y_tr)}
p3 = np.empty_like(y_te)
for r in range(3):
    b = ols(Xb_tr[reg_tr == r], y_tr[reg_tr == r])
    p3[reg_te == r] = Xb_te[reg_te == r] @ b
pred['B3_stratified'] = p3
mse = {k: float(((y_te - v) ** 2).mean()) for k, v in pred.items()}
print({k: round(v, 6) for k, v in mse.items()})
assert round(mse['B1_pooled'], 6) == SCREEN_B1_MSE, \
    f'B1 cross-check failed: {mse["B1_pooled"]:.6f} != {SCREEN_B1_MSE}'
print('B1 reproduces RV-S1 Gate 1 to 6 dp — cross-check PASS')

# ---- 3. M2 port (bundle code, soft_only headline mode; n_x = 1) -------------
class PerEdgeNetM2(nn.Module):
    def __init__(self, L, hidden):
        super().__init__()
        self.lin1 = nn.Linear(L, hidden); self.lin2 = nn.Linear(hidden, 1)
    def forward(self, x):
        return self.lin2(F.relu(self.lin1(x))).squeeze(-1)

class M2_RegimeAware(nn.Module):
    def __init__(self, L, n_x, hidden, n_regimes=3, tau_init=TAU_INIT,
                 frozen_centers=(0.2, 0.5, 0.8)):
        super().__init__()
        self.n_regimes, self.n_x, self.routing = n_regimes, n_x, 'soft'
        self.f_yy = nn.ModuleList([PerEdgeNetM2(L, hidden) for _ in range(n_regimes)])
        self.f_xy = nn.ModuleList([nn.ModuleList([PerEdgeNetM2(L, hidden)
                                   for _ in range(n_x)]) for _ in range(n_regimes)])
        self.bias = nn.Parameter(torch.zeros(n_regimes))
        self.register_buffer('centers', torch.tensor(frozen_centers, dtype=torch.float32))
        self.log_tau = nn.Parameter(torch.tensor(np.log(tau_init), dtype=torch.float32))
    def gating(self, psi):
        tau = torch.exp(self.log_tau)
        d2 = (psi.unsqueeze(-1) - self.centers.unsqueeze(0)) ** 2
        return F.softmax(-d2 / tau, dim=-1)
    def per_regime_outputs(self, Yh, Xh):
        outs = []
        for r in range(self.n_regimes):
            o = self.bias[r] + self.f_yy[r](Yh[:, :, 0])
            for j in range(self.n_x):
                o = o + self.f_xy[r][j](Xh[:, :, j])
            outs.append(o)
        return torch.stack(outs, dim=-1)
    def forward(self, Yh, Xh):
        pi = self.gating(Yh[:, -1, 0])
        return (pi * self.per_regime_outputs(Yh, Xh)).sum(dim=-1)

def per_regime_predictor_fn(model, r):
    def fn(Yh, Xh):
        o = model.bias[r] + model.f_yy[r](Yh[:, :, 0])
        for j in range(model.n_x):
            o = o + model.f_xy[r][j](Xh[:, :, j])
        return o
    return fn

def jacobian_softmax_norm_Y(model_fn, Yh_b, Xh_b, tau_softmax=5.0):
    Yh_in = Yh_b.clone().requires_grad_(True)
    out = model_fn(Yh_in, Xh_b)
    grad = torch.autograd.grad(out.sum(), Yh_in, create_graph=True)[0]
    norms = grad.flatten(1).norm(dim=1)
    return (torch.logsumexp(tau_softmax * norms, dim=0)
            - np.log(norms.shape[0])) / tau_softmax

def train_m2_soft_only(model, Yh, Xh, tgt, epochs, lr):
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=WD)
    Yh_t = torch.tensor(Yh, dtype=torch.float32, device=DEV)
    Xh_t = torch.tensor(Xh, dtype=torch.float32, device=DEV)
    tg_t = torch.tensor(tgt, dtype=torch.float32, device=DEV)
    N = len(Yh_t)
    for ep in range(epochs):
        perm = torch.randperm(N, device=DEV)
        for s in range(0, N, BATCH):
            idx = perm[s:s + BATCH]
            opt.zero_grad()
            loss = F.mse_loss(model(Yh_t[idx], Xh_t[idx]), tg_t[idx])
            spec = torch.zeros((), device=DEV)
            for r in range(model.n_regimes):
                sig = jacobian_softmax_norm_Y(per_regime_predictor_fn(model, r),
                                              Yh_t[idx], Xh_t[idx])
                spec = spec + F.relu(sig - TGT_LY) ** 2
            loss = loss + LAM_Y * spec + LAM_PHI * torch.exp(-model.log_tau) ** 2
            loss.backward(); opt.step()
    model.eval(); return model

def predict(model, Yh, Xh):
    with torch.no_grad():
        return model(torch.tensor(Yh, dtype=torch.float32, device=DEV),
                     torch.tensor(Xh, dtype=torch.float32, device=DEV)).cpu().numpy()

def run_m2(tag, Yh, Xh, tgt, Yh_ev, Xh_ev, y_ev, seed, lr, hid):
    ck = f'{CKPT}/{tag}.pt'
    if os.path.exists(ck):
        d = torch.load(ck, map_location='cpu', weights_only=False)
        resume_log.append(tag)
        return d['pred_ev'], d
    torch.manual_seed(seed); np.random.seed(seed)
    m = M2_RegimeAware(P_LAGS, 1, hid, frozen_centers=tuple(centers)).to(DEV)
    t0 = time.time()
    train_m2_soft_only(m, Yh, Xh, tgt, EPOCHS, lr)
    p_ev = predict(m, Yh_ev, Xh_ev)
    d = dict(pred_ev=p_ev, mse_ev=float(((y_ev - p_ev) ** 2).mean()),
             seed=seed, lr=lr, hidden=hid, minutes=(time.time() - t0) / 60,
             state=m.cpu().state_dict())
    torch.save(d, ck)
    return p_ev, d

# ---- 3a. Pre-registered grid validation (test data untouched) ---------------
tr_days = np.where(train_mask_day)[0]
n_val_days = round(0.2 * len(tr_days))
sub_mask = train_mask_day.copy(); sub_mask[tr_days[-n_val_days:]] = False
val_mask = np.zeros_like(train_mask_day); val_mask[tr_days[-n_val_days:]] = True
Yh_s, Xh_s, y_s, *_ = build(sub_mask)
Yh_v, Xh_v, y_v, *_ = build(val_mask)
val_table = []
for lr, hid in GRID:
    cell_mses = []
    for seed in SEEDS:
        _, d = run_m2(f'val_lr{lr}_h{hid}_s{seed}', Yh_s, Xh_s, y_s,
                      Yh_v, Xh_v, y_v, seed, lr, hid)
        cell_mses.append(d['mse_ev'])
    val_table.append(dict(lr=lr, hidden=hid, val_mse_mean=float(np.mean(cell_mses)),
                          val_mses=[round(m, 6) for m in cell_mses]))
    print(f'grid lr={lr} hidden={hid}: val MSE mean {np.mean(cell_mses):.6f}')
best = min(val_table, key=lambda r: r['val_mse_mean'])
LR_SEL, HID_SEL = best['lr'], best['hidden']
print(f'SELECTED: lr={LR_SEL} hidden={HID_SEL} (deterministic rule: min mean val MSE)')

# ---- 3b. Final training: full training set, 5 seeds -------------------------
m2_seed_preds, final_runs = [], []
for seed in SEEDS:
    p, d = run_m2(f'final_s{seed}', Yh_tr, Xh_tr, y_tr,
                  Yh_te, Xh_te, y_te, seed, LR_SEL, HID_SEL)
    m2_seed_preds.append(p); final_runs.append(d)
m2_ens = np.mean(m2_seed_preds, axis=0)
pred['M2_ens'] = m2_ens
mse['M2_ens'] = float(((y_te - m2_ens) ** 2).mean())
print(f"M2 [ENS] test MSE {mse['M2_ens']:.6f}  "
      f"per-seed {[round(float(((y_te-p)**2).mean()),6) for p in m2_seed_preds]}")

# ---- 4. Tests (time-block bootstrap, identical machinery to RV-S1) ----------
rng = np.random.default_rng(RNG_SEED_TESTS)
tds = np.sort(np.unique(day_te)); day_pos = {d: i for i, d in enumerate(tds)}
pos_te = np.array([day_pos[d] for d in day_te])

def day_mean(d_obs, mask=None):
    m = np.ones(len(d_obs), bool) if mask is None else mask
    s = np.zeros(n_test_days); c = np.zeros(n_test_days)
    np.add.at(s, pos_te[m], d_obs[m]); np.add.at(c, pos_te[m], 1)
    keep = c > 0
    return s[keep] / c[keep]

def block_boot_p(series, B, rng):
    def st(x):
        nf = (len(x) // BLOCK) * BLOCK
        if nf < 2 * BLOCK: return x.mean(), x.std(ddof=1) / np.sqrt(len(x))
        bm = x[:nf].reshape(-1, BLOCK).mean(1)
        return x.mean(), bm.std(ddof=1) / np.sqrt(len(bm))
    mo, so = st(series); t_obs = mo / so
    nblk = int(np.ceil(len(series) / BLOCK))
    starts = rng.integers(0, max(1, len(series) - BLOCK + 1), size=(B, nblk))
    idx = (starts[:, :, None] + np.arange(BLOCK)[None, None, :]).reshape(B, -1)[:, :len(series)]
    idx = np.minimum(idx, len(series) - 1)
    res = series[idx]; mb = res.mean(1)
    nf = (res.shape[1] // BLOCK) * BLOCK
    bm = res[:, :nf].reshape(B, -1, BLOCK).mean(2)
    sb = bm.std(1, ddof=1) / np.sqrt(bm.shape[1]); ok = sb > 0
    tb = (mb[ok] - mo) / sb[ok]
    return mo, t_obs, (1 + (np.abs(tb) >= abs(t_obs)).sum()) / (ok.sum() + 1)

def hac_dm(d_obs, mask=None):
    m = np.ones(len(d_obs), bool) if mask is None else mask
    out = []
    for j, a in enumerate(ASSETS):
        d = d_obs[m & (ast_te == j)]
        if len(d) < 2 * BLOCK: out.append(f'{a}:n/a'); continue
        dc = d - d.mean(); lrv = np.mean(dc ** 2)
        for l in range(1, BLOCK + 1):
            lrv += 2 * (1 - l / (BLOCK + 1)) * np.mean(dc[l:] * dc[:-l])
        out.append(f'{a}:{d.mean()/np.sqrt(max(lrv,1e-300)/len(d)):+.2f}')
    return '|'.join(out)

SCOPES = [('overall', None)] + [(f'regime{r}', reg_te == r) for r in range(3)]
FAMS = [('F1_M2_vs_B2interaction', 'B2_interaction', 'M2_ens', 'DECISIVE'),
        ('F2_M2_vs_B1pooled',      'B1_pooled',      'M2_ens', 'primary'),
        ('F3_B2_vs_B1',            'B1_pooled',      'B2_interaction', 'context'),
        ('F4_M2_vs_rw',            'rw',             'M2_ens', 'context')]
rows = []
for fam, base, mdl, role in FAMS:
    d_full = (y_te - pred[base]) ** 2 - (y_te - pred[mdl]) ** 2   # + favors mdl
    for scope, mask in SCOPES:
        mo, t_o, p = block_boot_p(day_mean(d_full, mask), B_BOOT, rng)
        seed_ts = ''
        if mdl == 'M2_ens':
            ts = []
            for sp in m2_seed_preds:
                ds = (y_te - pred[base]) ** 2 - (y_te - sp) ** 2
                sm, st_, _ = block_boot_p(day_mean(ds, mask), 1, np.random.default_rng(0))
                ts.append(st_)
            agree = sum(np.sign(t) == np.sign(t_o) for t in ts if np.isfinite(t))
            seed_ts = '|'.join(f'{t:+.2f}' for t in ts) + f'  [{agree}/5]'
        rows.append(dict(family=fam, role=role, scope=scope,
                         N=int(len(d_full) if mask is None else mask.sum()),
                         mse_base=float(((y_te - pred[base])**2)[mask].mean()
                                        if mask is not None else mse[base]),
                         mse_model=float(((y_te - pred[mdl])**2)[mask].mean()
                                         if mask is not None else mse[mdl]),
                         mean_d=mo, t_boot=t_o, p_boot=p,
                         low_power_flag=(scope == 'regime2'),
                         per_seed_t=seed_ts, per_asset_hac=hac_dm(d_full, mask)))
res = pd.DataFrame(rows)
res['p_holm'] = np.nan
for fam in ['F1_M2_vs_B2interaction', 'F2_M2_vs_B1pooled']:
    m = res.index[(res.family == fam) & (res.scope != 'overall')]
    pv = res.loc[m, 'p_boot'].to_numpy(); order = np.argsort(pv)
    adj = np.empty(3); running = 0.0
    for rank, oi in enumerate(order):
        running = max(running, (3 - rank) * pv[oi]); adj[oi] = min(1.0, running)
    res.loc[m, 'p_holm'] = adj
res.to_csv(f'{OUT_DIR}/results_RVF1_dm_tests.csv', index=False)
print('\n================ RV-F1 TESTS ================')
with pd.option_context('display.width', 240, 'display.max_columns', None):
    print(res.drop(columns=['per_asset_hac']).to_string(index=False))

def ov(fam):
    r = res[(res.family == fam) & (res.scope == 'overall')].iloc[0]
    return r.p_boot < ALPHA and r.mean_d > 0
level = 'A' if ov('F1_M2_vs_B2interaction') else ('B' if ov('F2_M2_vs_B1pooled') else 'C')
print(f'\nPre-specified claim level: {level}')

# ---- 5. Certificate audit on the 5 final seeds ------------------------------
def full_audit(model):
    model = model.cpu().eval()
    Yt = torch.tensor(Yh_te, dtype=torch.float32)
    Xt = torch.tensor(Xh_te, dtype=torch.float32)
    psi_te = Yh_te[:, -1, 0]
    L_Y = np.zeros(3); arng = np.random.default_rng(0)
    sidx = arng.choice(len(Yt), size=min(200, len(Yt)), replace=False)
    for r in range(3):
        mx = 0.0
        for i in sidx:
            Yh1 = Yt[i:i+1].clone().requires_grad_(True)
            pr = model.bias[r] + model.f_yy[r](Yh1[:, :, 0]) \
                 + model.f_xy[r][0](Xt[i:i+1][:, :, 0])
            g = torch.autograd.grad(pr.sum(), Yh1)[0]
            mx = max(mx, float(g.flatten().norm()))
        L_Y[r] = mx
    L_max = float(L_Y.max())
    with torch.no_grad():
        per_r = model.per_regime_outputs(Yt, Xt).numpy()
        pi_te = model.gating(torch.tensor(psi_te, dtype=torch.float32)).numpy()
    M = max(float(np.abs(per_r[:, r1] - per_r[:, r2]).max())
            for r1 in range(3) for r2 in range(r1 + 1, 3))
    def max_slope(lo, hi, n):
        grid = torch.linspace(float(lo), float(hi), n)
        with torch.no_grad():
            pi = model.gating(grid).numpy()
        dpsi = float(grid[1] - grid[0])
        return float(np.abs(np.diff(pi, axis=0)).sum(1).max() / dpsi)
    L_phi = max_slope(psi_te.min(), psi_te.max(), 500)
    L_phi_eq = np.zeros(3)
    masks = [psi_te < cuts[0], (psi_te >= cuts[0]) & (psi_te < cuts[1]), psi_te >= cuts[1]]
    for r, mk in enumerate(masks):
        if mk.sum() < 10: L_phi_eq[r] = np.nan; continue
        lo, hi = np.percentile(psi_te[mk], [25, 75])
        L_phi_eq[r] = max_slope(lo, hi, 50)
    L_eq = L_Y + L_phi_eq * M
    L_eq_max = float(np.nanmax(L_eq)); B_trans = L_max + L_phi * M
    r_pred = pi_te.argmax(1); alphas = []
    for u in range(A):
        ru = r_pred[ast_te == u]
        if len(ru) > 1: alphas.append((ru[1:] != ru[:-1]).sum() / (len(ru) - 1))
    alpha_emp = float(np.mean(alphas))
    alpha_star = (None if L_eq_max >= 1 else
                  float('inf') if B_trans <= 1 else
                  float(-np.log(L_eq_max) / (np.log(B_trans) - np.log(L_eq_max))))
    rho = None if L_eq_max >= 1 else float(L_eq_max**(1-alpha_emp) * B_trans**alpha_emp)
    return dict(L_Y=[round(float(x), 4) for x in L_Y], L_max=round(L_max, 4),
                M=round(M, 4), tau=round(float(torch.exp(model.log_tau).detach()), 4),
                L_phi=round(L_phi, 4),
                L_phi_eq_max=round(float(np.nanmax(L_phi_eq)), 4),
                L_eq_max=round(L_eq_max, 4), B_trans=round(B_trans, 4),
                alpha_emp=round(alpha_emp, 4),
                alpha_star=(None if alpha_star is None else round(alpha_star, 4)),
                rho=(None if rho is None else round(rho, 4)),
                A3_nontrivial=bool(L_eq_max < 1 and B_trans > 1))

audit_rows = []
for seed in SEEDS:
    d = torch.load(f'{CKPT}/final_s{seed}.pt', map_location='cpu', weights_only=False)
    m = M2_RegimeAware(P_LAGS, 1, d['hidden'], frozen_centers=tuple(centers))
    m.load_state_dict(d['state'])
    audit_rows.append(dict(panel='RV', seed=seed, **full_audit(m)))
# V-Dem reference row from the frontier artifact (verify-from-artifact, no memory)
import pickle
fb = open(FRONTIER_PATH, 'rb').read()
assert hashlib.sha256(fb).hexdigest() == FRONTIER_SHA
fr = pickle.loads(fb)
vd = fr['results'][('soft_only', 0.0)]
audit_rows.append(dict(panel='VDem_ref(soft_only,0.0)', seed='mean_5',
                       L_max=round(float(np.mean([r['L_max'] for r in vd])), 4),
                       M=round(float(np.mean([r['M'] for r in vd])), 4),
                       L_eq_max=round(float(np.mean([r['L_eq_max'] for r in vd])), 4),
                       B_trans=round(float(np.mean([r['B_trans'] for r in vd])), 4)))
aud = pd.DataFrame(audit_rows)
aud.to_csv(f'{OUT_DIR}/results_RVF1_audit_constants.csv', index=False)
print('\n================ CERTIFICATE AUDIT ================')
print(aud.to_string(index=False))

# ---- 6. Meta ----------------------------------------------------------------
meta = dict(design='design_RV_F1_ladder.md (locked 2026-07-25)',
            data_sha256=DATA_SHA, frontier_sha256=FRONTIER_SHA,
            grid=[dict(lr=g[0], hidden=g[1]) for g in GRID],
            validation_table=val_table, selected=dict(lr=LR_SEL, hidden=HID_SEL),
            epochs=EPOCHS, batch=BATCH, seeds=SEEDS,
            tau_init=TAU_INIT, lam_phi=LAM_PHI, lam_Y=LAM_Y, target_LY=TGT_LY,
            centers=[float(c) for c in centers], cuts=[float(c) for c in cuts],
            rng_seed_tests=RNG_SEED_TESTS, B=B_BOOT, block=BLOCK,
            baseline_mses={k: round(v, 6) for k, v in mse.items()},
            claim_level=level, resumed_checkpoints=resume_log,
            run_minutes=[round(r.get('minutes', 0), 1) for r in final_runs],
            device=DEV, torch=torch.__version__, numpy=np.__version__,
            timestamp_utc=datetime.datetime.now(datetime.timezone.utc).isoformat())
with open(f'{OUT_DIR}/results_RVF1_meta.json', 'w') as fh:
    json.dump(meta, fh, indent=2)
print(f'\nClaim level {level}. Wrote 3 result files + checkpoints to {OUT_DIR}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
device: cuda
regime cuts [-5.0229 -4.6019]  gate centers [-5.2478 -4.8221 -4.3146]
train obs 16704  test obs 4184  test regime counts [2919  989  276]
{'rw': 0.103898, 'B1_pooled': 0.07462, 'B2_interaction': 0.07464, 'B3_stratified': 0.075412}
B1 reproduces RV-S1 Gate 1 to 6 dp — cross-check PASS
grid lr=0.001 hidden=16: val MSE mean 0.084488
grid lr=0.001 hidden=32: val MSE mean 0.084668
grid lr=0.0003 hidden=16: val MSE mean 0.083530
grid lr=0.0003 hidden=32: val MSE mean 0.083928
SELECTED: lr=0.0003 hidden=16 (deterministic rule: min mean val MSE)
M2 [ENS] test MSE 0.075734  per-seed [0.076378, 0.07672, 0.075015, 0.0744, 0.077437]

================ RV-F1 TESTS ================
                family     role   scope    N  mse_base  mse_model    mean_d    t_boot  p_boot  low_power_flag                           per_seed_t  p_holm
F1_M2_vs_B2interaction DECI